In [ ]:
"""
Informe de Retroalimentación - Unidad 3
Nivel de Aprobación del Uso de Aula Virtual Moodle UAGRM
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager as fm

# ============================================================
# CONFIGURACIÓN
# ============================================================
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# Paleta de colores corporativos
AZUL = '#4A3FDB'
AZUL_CLARO = '#6B5FFF'
NARANJA = '#F5A623'
ROJO = '#E74C3C'
VERDE = '#2ECC71'
GRIS = '#7F8C8D'
BLANCO = '#FFFFFF'

# ============================================================
# CARGAR DATOS
# ============================================================
# Reemplaza esta ruta con la ubicación de tu archivo CSV
CSV_PATH = 'Retroalimentación Unidad 3 — Estadística II.csv'

df = pd.read_csv(CSV_PATH)
df.columns = ['Marca_temporal', 'P1_Estructura', 'P2_Videos', 
              'P3_Division', 'P4_Dificil', 'P5_Tiempo', 'P6_Rubrica']

# ============================================================
# CÁLCULOS DE INDICADORES
# ============================================================
n_total = 17
n_p3 = 16  # una respuesta nula

p1_pos = ((df['P1_Estructura'] >= 3).sum() / n_total) * 100
p2_pos = ((df['P2_Videos'] >= 3).sum() / n_total) * 100
p3_pos = ((df['P3_Division'] >= 3).sum() / n_p3) * 100
p6_pos = ((df['P6_Rubrica'] >= 3).sum() / n_total) * 100
promedio_general = df[['P1_Estructura', 'P2_Videos', 'P3_Division', 'P6_Rubrica']].mean().mean()

# ============================================================
# CREAR FIGURA
# ============================================================
fig = plt.figure(figsize=(16, 22))
fig.patch.set_facecolor('#F8F9FA')

# Título
fig.suptitle('INFORME DE RETROALIMENTACION - UNIDAD 3\n'
             'Nivel de Aprobacion del Uso de Aula Virtual Moodle UAGRM', 
             fontsize=18, fontweight='bold', color=AZUL, y=0.97)
fig.text(0.5, 0.945, f'Encuesta aplicada a {n_total} estudiantes | Estadistica II | Junio 2026', 
         ha='center', fontsize=11, color=GRIS, style='italic')

# ============================================================
# PANEL KPIs
# ============================================================
ax_kpi = fig.add_axes([0.05, 0.87, 0.9, 0.06])
ax_kpi.set_facecolor('#F8F9FA')
ax_kpi.axis('off')

kpis = [
    (f'{p1_pos:.0f}%', 'Estructura\nclara', VERDE),
    (f'{p2_pos:.0f}%', 'Videos\nutiles', VERDE),
    (f'{p3_pos:.0f}%', 'Division\nclara', VERDE),
    (f'{p6_pos:.0f}%', 'Rubrica\nclara', VERDE),
    (f'{promedio_general:.2f}', 'Promedio\nGeneral (1-4)', AZUL),
]

for i, (val, label, color) in enumerate(kpis):
    x = 0.1 + i * 0.18
    rect = mpatches.FancyBboxPatch((x, 0.1), 0.14, 0.8, 
                                    boxstyle="round,pad=0.02", 
                                    facecolor=color, edgecolor='none', alpha=0.15)
    ax_kpi.add_patch(rect)
    ax_kpi.text(x + 0.07, 0.6, val, ha='center', va='center', 
                fontsize=16, fontweight='bold', color=color)
    ax_kpi.text(x + 0.07, 0.25, label, ha='center', va='center', 
                fontsize=9, color=GRIS)

# ============================================================
# FUNCION AUXILIAR: Grafica de barras Likert
# ============================================================
def grafica_likert(ax, data, titulo, n_resp, ylim_max):
    ax.set_facecolor(BLANCO)
    for spine in ax.spines.values():
        spine.set_visible(False)
    
    counts = data.value_counts().sort_index()
    labels = ['1\nMuy mal', '2\nMal', '3\nBien', '4\nMuy bien']
    colors = [ROJO, NARANJA, AZUL_CLARO, AZUL]
    
    bars = ax.bar(range(4), [counts.get(i, 0) for i in range(1, 5)], 
                  color=colors, width=0.6, edgecolor='white', linewidth=2)
    
    for i, bar in enumerate(bars):
        h = bar.get_height()
        if h > 0:
            pct = (h / n_resp) * 100
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, 
                   f'{int(h)} ({pct:.1f}%)', 
                   ha='center', va='bottom', fontsize=10, 
                   fontweight='bold', color=AZUL)
    
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, ylim_max)
    ax.set_title(titulo, fontsize=11, fontweight='bold', 
                color=AZUL, pad=10, loc='left')
    ax.text(0.02, 0.95, f'{n_resp} respuestas', 
            transform=ax.transAxes, fontsize=8, color=GRIS, va='top')
    ax.tick_params(left=False, labelleft=False)
    ax.set_yticks([])

# ============================================================
# GRAFICAS 1-4: Barras Likert
# ============================================================
grafica_likert(fig.add_axes([0.05, 0.67, 0.42, 0.18]), 
               df['P1_Estructura'], 
               '1. Estructura clara e intuitiva de las lecciones', 17, 13)

grafica_likert(fig.add_axes([0.53, 0.67, 0.42, 0.18]), 
               df['P2_Videos'], 
               '2. Videos complementarios utiles', 17, 10)

grafica_likert(fig.add_axes([0.05, 0.445, 0.42, 0.18]), 
               df['P3_Division'], 
               '3. Claridad de la division del practico (60% Cuestionario + 40% PDF)', 16, 10)

grafica_likert(fig.add_axes([0.53, 0.445, 0.42, 0.18]), 
               df['P6_Rubrica'], 
               '6. Rubrica de la constancia PDF clara antes de entregar', 17, 10)

# ============================================================
# GRAFICA 5: Dificultad (Pie)
# ============================================================
ax5 = fig.add_axes([0.05, 0.22, 0.42, 0.18])
ax5.set_facecolor(BLANCO)
for spine in ax5.spines.values():
    spine.set_visible(False)

p4_counts = df['P4_Dificil'].value_counts()
colors_p4 = [AZUL, NARANJA, VERDE]
wedges, texts, autotexts = ax5.pie(p4_counts.values, labels=p4_counts.index, 
                                     autopct='%1.1f%%', startangle=90, colors=colors_p4,
                                     textprops={'fontsize': 10, 'color': BLANCO, 'fontweight': 'bold'},
                                     pctdistance=0.6)
for autotext in autotexts:
    autotext.set_color(BLANCO)
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)
for text in texts:
    text.set_color(GRIS)
    text.set_fontsize(9)

ax5.set_title('4. Que parte del practico resulto mas dificil?', 
              fontsize=11, fontweight='bold', color=AZUL, pad=10, loc='left')
ax5.text(-0.15, 1.05, '17 respuestas', transform=ax5.transAxes, fontsize=8, color=GRIS, va='top')

# ============================================================
# GRAFICA 6: Tiempo (Pie)
# ============================================================
ax6 = fig.add_axes([0.53, 0.22, 0.42, 0.18])
ax6.set_facecolor(BLANCO)
for spine in ax6.spines.values():
    spine.set_visible(False)

p5_counts = df['P5_Tiempo'].value_counts()
colors_p5 = [AZUL, NARANJA]
wedges, texts, autotexts = ax6.pie(p5_counts.values, labels=p5_counts.index, 
                                     autopct='%1.1f%%', startangle=90, colors=colors_p5,
                                     textprops={'fontsize': 10, 'color': BLANCO, 'fontweight': 'bold'},
                                     pctdistance=0.6)
for autotext in autotexts:
    autotext.set_color(BLANCO)
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)
for text in texts:
    text.set_color(GRIS)
    text.set_fontsize(9)

ax6.set_title('5. Tiempo para completar el practico', 
              fontsize=11, fontweight='bold', color=AZUL, pad=10, loc='left')
ax6.text(-0.15, 1.05, '17 respuestas', transform=ax6.transAxes, fontsize=8, color=GRIS, va='top')

# ============================================================
# INFORME EJECUTIVO
# ============================================================
ax_text = fig.add_axes([0.05, 0.02, 0.9, 0.17])
ax_text.set_facecolor('#F0F0FF')
for spine in ax_text.spines.values():
    spine.set_visible(False)
ax_text.axis('off')

informe = """
INFORME EJECUTIVO - RECOMENDACION SOBRE CONTINUAR CON EL AULA VIRTUAL MOODLE UAGRM

RESULTADOS CLAVE:
  * El 94.1% de los estudiantes considera que el tiempo para completar el practico fue ADECUADO.
  * Entre 88.2% y 93.8% de los estudiantes califican con 3 o 4 (Bien/Muy bien) las 4 dimensiones evaluadas.
  * El promedio general de satisfaccion es 3.43 sobre 4.0, lo que indica una percepcion MUY POSITIVA.
  * Solo 1 estudiante (5.9%) reporto dificultades con el tiempo; ninguno califico con 1 (Muy mal).

AREAS DE ATENCION:
  * 35.3% de los estudiantes reporto "Subir el PDF" como la parte mas dificil (problema tecnico, no pedagogico).
  * 29.4% reporto dificultades con "El cuestionario" (posiblemente la dificultad del contenido matematico).
  * 1 respuesta nula en la pregunta 3 (division del practico), lo que sugiere que algunos no entendieron la pregunta.

RECOMENDACION ESTRATEGICA: SI, CONTINUAR CON EL AULA VIRTUAL MOODLE UAGRM

  Los datos demuestran que el aula virtual ha sido BIEN RECIBIDA por los estudiantes. La percepcion general es
  positiva y el tiempo de dedicacion es adecuado. Se recomienda:

  1. MANTENER la estructura de lecciones 3.1, 3.2, 3.3 (alta aprobacion: 94.1% con 3 o 4).
  2. CONTINUAR con videos complementarios (94.1% los consideran utiles).
  3. MEJORAR el tutorial de "Subir el PDF" (principal dificultad reportada, 35.3%).
  4. REFORZAR la explicacion de la division 60%/40% del practico (algunos estudiantes no respondieron).
  5. MANTENER el formato de evaluacion actual (cuestionario + constancia PDF), ya que el tiempo es adecuado.
"""

ax_text.text(0.02, 0.98, informe, transform=ax_text.transAxes, fontsize=9.5, 
             color='#2C3E50', va='top', ha='left', linespacing=1.5, family='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#F0F0FF', 
                      edgecolor=AZUL, alpha=0.3, linewidth=2))

# Guardar
plt.savefig('Informe_Retroalimentacion_Unidad3.pdf', 
            format='pdf', dpi=300, bbox_inches='tight', facecolor='#F8F9FA')
plt.savefig('Informe_Retroalimentacion_Unidad3.png', 
            format='png', dpi=300, bbox_inches='tight', facecolor='#F8F9FA')
print("Informe generado: Informe_Retroalimentacion_Unidad3.pdf")
print("Imagen generada: Informe_Retroalimentacion_Unidad3.png")
plt.show()